# makemore_wavenet

## preparation

import code from lec04

In [1]:
# import module
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt # for making figures
%matplotlib inline

In [2]:
# read file
words = open('../names.txt', 'r').read().splitlines()

In [3]:
# utility functions
from string import ascii_lowercase
stoi = {c:i+1 for i, c in enumerate(ascii_lowercase)}
stoi['.'] = 0
itos = {i:c for c, i in stoi.items()}
vocab_size = len(itos)

In [4]:
# build and split the dataset
block_size = 3

def build_dataset(words):  
    X, Y = [], []
    for w in words:
        context = [0] * block_size
        for ch in w + '.':
            ix = stoi[ch]
            X.append(context)
            Y.append(ix)
            context = context[1:] + [ix]
    X = torch.tensor(X)
    Y = torch.tensor(Y)
    print(X.shape, Y.shape)
    return X, Y

import random
random.seed(42)
random.shuffle(words)
n1 = int(0.8 * len(words))
n2 = int(0.9 * len(words))

Xtr, Ytr = build_dataset(words[:n1])        # 80% training set
Xdev, Ydev = build_dataset(words[n1:n2])    # 10% dev set
Xte, Yte = build_dataset(words[n2:])        # 10% test set

torch.Size([182625, 3]) torch.Size([182625])
torch.Size([22655, 3]) torch.Size([22655])
torch.Size([22866, 3]) torch.Size([22866])


build same API as nn.Module in PyTorch

In [ ]:
# PyTorchifying the code (mimicking the nn.Module API):
# These classes replicate the standard PyTorch interface, eliminating boilerplate 
# and allowing us to build neural networks declaratively.
# 
# - `__init__`   : Initializes the layer instance.
# - `__call__`   : Executes the forward pass.
# - `parameters` : Returns the learnable parameters (for the optimizer).

class Linear:
    def __init__(self, fan_in, fan_out, bias=True):
        self.weights = torch.randn((fan_in, fan_out)) / fan_in**0.5
        self.biases = torch.zeros((fan_out)) if bias else None

    def __call__(self, x):
        self.out = x @ self.weights
        if self.biases is not None:
            self.out += self.biases
        return self.out

    def parameters(self):
        return [self.weights] + ([self.biases] if self.biases is not None else [])


class BatchNorm1d:
    def __init__(self, dim, eps=1e-5, momentum=0.1):
        self.eps = eps
        self.momentum = momentum
        self.training = True
        self.gamma = torch.ones(dim)
        self.beta = torch.zeros(dim)
        self.running_mean = torch.zeros(dim)
        # Note: As per the original paper, we track the running variance here 
        # rather than the running std used in Lec04.
        self.running_var = torch.ones(dim)

    def __call__(self, x):
        if self.training:
            self.bnmean = x.mean(dim=0, keepdim=True)
            self.bnvar = x.var(dim=0, keepdim=True)
        else:
            self.bnmean = self.running_mean
            self.bnvar = self.running_var
        
        bnhat = (x - self.bnmean) * torch.sqrt(self.bnvar+self.eps)**(-1)
        self.out = self.gamma * bnhat + self.beta

        if self.training:
            with torch.no_grad():
                self.running_mean = (1-self.momentum)*self.running_mean + self.momentum*self.bnmean
                self.running_var = (1-self.momentum)*self.running_var + self.momentum*self.bnvar
        
        return self.out
    
    def parameters(self):
        return [self.gamma, self.beta]


class Tanh:
    def __call__(self, x):
        self.out = torch.tanh(x)
        return self.out
    
    def parameters(self):
        return []

In [ ]:
torch.manual_seed(42)    # seed rng for reproducibility

In [ ]:
# build neural network
emb_dim = 10    # dimension of embedding vectors
ns_num_h = 200    # the number of neurons in hidden layer

C = torch.randn((vocab_size, emb_dim))
Layers = [
    Linear(emb_dim*block_size, ns_num_h, bias=False), BatchNorm1d(ns_num_h), Tanh(),
    Linear(ns_num_h, vocab_size)
]

with torch.no_grad():
    Layers[-1].weights *= 0.1

parameters = [C] + [p for l in Layers for p in l.parameters()]
for p in parameters:
    p.requires_grad = True

In [ ]:
# train
max_steps = 200000
batch_size = 32

# for layer in Layers:
#     if isinstance(layer, BatchNorm1d):
#         layer.training = True

for k in range(max_steps):

    # mini-batch
    ix = torch.randint(0, Xtr.shape[0], (batch_size,))
    Xb, Yb = Xtr[ix], Ytr[ix]

    # forward pass
    emb = C[Xb]
    x = emb.view(-1, emb_dim*block_size)
    for layer in Layers:
        x = layer(x)
    loss = F.cross_entropy(x, Yb)

    # backward pass
    for p in parameters:
        p.grad = None
    loss.backward()

    # update
    lr = 0.1 if k < 100000 else 0.01
    for p in parameters:
        p.data += -p.grad * lr

In [ ]:
# evaluate the model and adjust hyperparameters
@torch.no_grad()
def split_loss(split):
    Xb, Yb = {
        'tr': (Xtr, Ytr),
        'dev': (Xdev, Ydev),
        'te': (Xte, Yte)
    }[split]

    emb = C[Xb]
    x = emb.view(-1, emb_dim*block_size)
    for layer in Layers:
        x = layer(x)
    loss = F.cross_entropy(x, Yb)
    print(f"{split} loss: {loss.data:.4f}")
    return loss.data

for layer in Layers:
    if isinstance(layer, BatchNorm1d):
        layer.training = False
tr_loss = split_loss('tr')
dev_loss = split_loss('dev')

tr loss: 2.0634
dev loss: 2.1080
